In [1]:
import os, json, math, random, gc, time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score
print("✅ Imports successful")

✅ Imports successful


# BirdCLEF 2026 Training - Phase 1 Improvements
- Light augmentation (time masking + frequency masking)
- Per-species optimal threshold computation from validation data

In [2]:
print(torch.cuda.is_available())

True


In [3]:
# === DATA PATHS ===
TRAIN_META_CSV = "/kaggle/input/competitions/birdclef-2026/train.csv"
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/train_audio"

df = pd.read_csv(TRAIN_META_CSV)
species = sorted(df["primary_label"].astype(str).unique())
species_set = set(species)

print(f"Number of species: {len(species)}")
print(f"Species: {species[:20]}")

idx = {lab: i for i, lab in enumerate(species)}

# Save species list
with open("/kaggle/working/species.json", "w") as f:
    json.dump(species, f)

print("✅ Species saved")

Number of species: 206
Species: ['1161364', '116570', '1176823', '1595929', '209233', '22930', '22956', '22961', '22967', '22973', '22983', '22985', '23150', '23154', '23158', '23176', '23724', '24279', '24285', '24287']
✅ Species saved


In [4]:
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ("", "[]"): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except:
        return []

def row_to_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(len(species), dtype="float32")
    p = str(primary_id)
    if p in idx: y[idx[p]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in species_set:
            y[idx[sid]] = 1.0
    return y

import ast

CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
    epochs=15,
    lr=1e-3,
    num_workers=4,
    persistent_workers=True,
    prefetch_factor=4,
    pin_memory=False,
    seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Config device: {CFG['device']}")

Config device: cuda


In [5]:
# PRECOMPUTE MELS (RUN ONCE)
import os, librosa, soundfile as sf
import numpy as np
from tqdm import tqdm
from pathlib import Path

MEL_OUT_DIR = "/kaggle/working/mels"
os.makedirs(MEL_OUT_DIR, exist_ok=True)

def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target-len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr//2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S01 = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)
    return S01.astype(np.float32)

print("Precomputing mels from train_audio…")

for idx_, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
    except:
        print(f"Failed: {audio_path}")
        continue

    if sr0 != CFG["sr"]:
        y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])

    y = fixed_length_mono(y, CFG["sr"], CFG["seconds"])
    mel = logmel_from_wave(y, CFG["sr"])

    np.save(
        Path(MEL_OUT_DIR) / (row["filename"].replace("/", "_") + ".npy"),
        mel
    )

print(f"✅ Mels saved from train_audio: {MEL_OUT_DIR}")

# =========================================================
# AUGMENT WITH TRAIN_SOUNDSCAPES (for species lacking train_audio data)
# =========================================================
print("\nLoading train_soundscapes labels...")
try:
    soundscape_labels = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")
    
    # Get species that appear in soundscapes
    soundscape_species = set()
    for species_list in soundscape_labels['primary_label']:
        species_list = str(species_list).split(';')
        for sp in species_list:
            soundscape_species.add(sp.strip())
    
    # Find species in soundscapes but NOT in train_audio
    missing_species = soundscape_species - species_set
    print(f"Species in soundscapes: {len(soundscape_species)}")
    print(f"Species missing from train_audio: {len(missing_species)}")
    
    if len(missing_species) > 0:
        print(f"Missing species: {sorted(missing_species)[:20]}...")
        
        # Extract training segments from soundscapes for missing species
        SOUNDSCAPES_DIR = "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
        count = 0
        
        # Helper function to convert HH:MM:SS to seconds
        def time_string_to_seconds(time_str):
            """Convert HH:MM:SS format to seconds"""
            parts = str(time_str).split(':')
            if len(parts) == 3:
                hours, minutes, seconds = map(int, parts)
                return hours * 3600 + minutes * 60 + seconds
            return 0
        
        for idx_, row in tqdm(soundscape_labels.iterrows(), total=len(soundscape_labels)):
            filename = row['filename']
            start_time_str = row['start']
            end_time_str = row['end']
            species_list = str(row['primary_label']).split(';')
            species_list = [sp.strip() for sp in species_list]
            
            # Only process if contains a missing species
            if not any(sp in missing_species for sp in species_list):
                continue
            
            audio_path = Path(SOUNDSCAPES_DIR) / filename
            if not audio_path.exists():
                continue
            
            try:
                y, sr0 = sf.read(audio_path, always_2d=False)
            except:
                continue
            
            # Convert time strings (HH:MM:SS) to seconds
            start_time_sec = time_string_to_seconds(start_time_str)
            end_time_sec = time_string_to_seconds(end_time_str)
            
            # Extract segment
            start_sample = int(start_time_sec * sr0)
            end_sample = int(end_time_sec * sr0)
            segment = y[start_sample:end_sample]
            
            # Resample if needed
            if sr0 != CFG["sr"]:
                segment = librosa.resample(segment, orig_sr=sr0, target_sr=CFG["sr"])
            
            # Enforce 5 seconds
            segment = fixed_length_mono(segment, CFG["sr"], CFG["seconds"])
            mel = logmel_from_wave(segment, CFG["sr"])
            
            # Save with unique name using original string timestamps
            mel_name = f"soundscape_{filename.rsplit(".", 1)[0]}_{start_time_str.replace(':', '')}_{end_time_str.replace(':', '')}.npy"
            np.save(Path(MEL_OUT_DIR) / mel_name, mel)
            count += 1
        
        print(f"✅ Extracted {count} segments from train_soundscapes for missing species")
    
except Exception as e:
    print(f"⚠️  Could not load train_soundscapes_labels: {e}")
    print("Proceeding with train_audio only")

print(f"✅ Total mels ready for training")


Precomputing mels from train_audio…


100%|██████████| 35549/35549 [44:57<00:00, 13.18it/s]


✅ Mels saved from train_audio: /kaggle/working/mels

Loading train_soundscapes labels...
Species in soundscapes: 75
Species missing from train_audio: 28
Missing species: ['1491113', '25073', '47158son01', '47158son02', '47158son03', '47158son04', '47158son05', '47158son06', '47158son07', '47158son08', '47158son09', '47158son10', '47158son11', '47158son12', '47158son13', '47158son14', '47158son15', '47158son16', '47158son17', '47158son18']...


100%|██████████| 1478/1478 [01:39<00:00, 14.84it/s]

✅ Extracted 1038 segments from train_soundscapes for missing species
✅ Total mels ready for training


In [6]:
# DATASET WITH AUGMENTATION
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path
import pandas as pd

MEL_ROOT = "/kaggle/working/mels"

class ClipDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, mel_root: str, cfg: dict, train: bool):
        self.df = frame.reset_index(drop=True)
        self.mel_root = Path(mel_root)
        self.cfg = cfg
        self.train = train
        self.win_frames = int(cfg["seconds"] * cfg["sr"] / cfg["hop"]) + 1

    def apply_light_augmentation(self, mel):
        """Apply light time-frequency masking augmentation for training"""
        if not self.train:
            return mel
        
        # Light time masking (30% chance)
        if np.random.rand() < 0.3:
            mask_width = np.random.randint(5, 15)
            mask_start = np.random.randint(0, max(1, mel.shape[1] - mask_width))
            mel[:, mask_start:mask_start+mask_width] *= np.random.uniform(0.1, 0.5)
        
        # Light frequency masking (30% chance)
        if np.random.rand() < 0.3:
            mask_height = np.random.randint(2, 8)
            mask_start = np.random.randint(0, max(1, mel.shape[0] - mask_height))
            mel[mask_start:mask_start+mask_height, :] *= np.random.uniform(0.1, 0.5)
        
        return mel

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]

        mel_name = r["filename"].replace("/", "_") + ".npy"
        mel_path = self.mel_root / mel_name
        mel = np.load(mel_path).astype("float32")

        T = mel.shape[1]
        W = self.win_frames

        if T <= W:
            pad = np.zeros((mel.shape[0], W - T), dtype=np.float32)
            mel = np.concatenate([mel, pad], axis=1)
        else:
            start = np.random.randint(0, T - W) if self.train else max(0, (T - W) // 2)
            mel = mel[:, start:start + W]
        
        # Apply light augmentation
        mel = self.apply_light_augmentation(mel)
        mel = np.clip(mel, 0.0, 1.0).astype(np.float32)

        x = torch.from_numpy(mel)[None, ...]
        y = torch.from_numpy(row_to_multihot(r["primary_label"], r["secondary_labels"])).float()

        return x.float(), y

print("✅ Dataset class defined with augmentation")


✅ Dataset class defined with augmentation


In [7]:
# AUGMENT TRAINING DATA WITH SOUNDSCAPE SEGMENTS
print("Augmenting training data with soundscape segments...")

try:
    soundscape_labels = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")
    
    # Get species that appear in soundscapes
    soundscape_species = set()
    for species_list in soundscape_labels['primary_label']:
        species_list = str(species_list).split(';')
        for sp in species_list:
            soundscape_species.add(sp.strip())
    
    # Find species in soundscapes but NOT in train_audio
    missing_species = soundscape_species - species_set
    
    # Convert to training format
    soundscape_rows = []
    for idx_, row in soundscape_labels.iterrows():
        filename = row['filename']
        species_str = row['primary_label']
        species_list = str(species_str).split(';')
        species_list = [sp.strip() for sp in species_list]
        
        # Only use if segment contains at least one missing species (same condition as Cell 5)
        if not any(sp in missing_species for sp in species_list):
            continue
        
        # Remove colons from time strings to match mel filename format
        start_time_clean = str(row['start']).replace(':', '')
        end_time_clean = str(row['end']).replace(':', '')
        
        soundscape_rows.append({
            'filename': f"soundscape_{filename.rsplit(".", 1)[0]}_{start_time_clean}_{end_time_clean}",
            'primary_label': species_list[0],  # Use first species as primary
            'secondary_labels': species_str,
            'rating': 5.0,  # Assume labeled data is high quality
            'author': 'competition',
            'collection': 'soundscape'
        })
    
    if len(soundscape_rows) > 0:
        soundscape_df = pd.DataFrame(soundscape_rows)
        
        # Append to training data
        df_augmented = pd.concat([df, soundscape_df], ignore_index=True)
        
        print(f"Original training samples: {len(df)}")
        print(f"Added soundscape segments: {len(soundscape_df)}")
        print(f"Total training samples: {len(df_augmented)}")
        
        # Use augmented dataset
        df = df_augmented
    else:
        print("No soundscape segments added (all species already in train_audio)")
        
except Exception as e:
    print(f"Could not load soundscape data: {e}")
    print("Proceeding with train_audio only")

print(f"Final dataset size: {len(df)} samples")



Augmenting training data with soundscape segments...
Original training samples: 35549
Added soundscape segments: 1038
Total training samples: 36587
Final dataset size: 36587 samples


In [8]:
# MODEL DEFINITION - ResNet50
import torch
import torch.nn as nn
from torchvision.models import resnet50

class ResNet50Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        self.n_mels = n_mels
        self.n_classes = n_classes
        
        # ResNet50 backbone
        self.model = resnet50(weights=None)
        
        # Modify first conv to accept 1 channel (mel spectrogram)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Replace fc with custom head for multi-label
        in_features = self.model.fc.in_features
        self.model.fc = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, n_classes)
        )
    
    def forward(self, x):
        # x: (batch, 1, n_mels, time_steps)
        features = self.model(x)  # (batch, in_features)
        logits = self.head(features)
        return logits

print("✅ ResNet50 model defined")


✅ ResNet50 model defined


In [9]:
from sklearn.model_selection import GroupKFold

groups = df["author"].fillna(df["collection"]).fillna(df["primary_label"]).astype(str)
gkf = GroupKFold(n_splits=5)

folds = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=groups)):
    folds.append((train_idx, val_idx))

print("✅ Created 5 folds")

✅ Created 5 folds


In [10]:
# RESNET18 MODEL
import torch
import torch.nn as nn
from torchvision.models import resnet18

class ResNet18Audio(nn.Module):
    def __init__(self, n_mels, n_classes):
        super().__init__()
        self.model = resnet18(weights=None)
        self.model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.model.fc = nn.Linear(self.model.fc.in_features, n_classes)

    def forward(self, x):
        return self.model(x)

print("✅ Model defined")

✅ Model defined


In [11]:
from sklearn.metrics import roc_auc_score
import numpy as np, time

@torch.no_grad()
def evaluate_macro_auc(model, dl, device):
    model.eval()
    all_logits, all_targets = [], []
    for x, y in dl:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        all_logits.append(logits.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())
    P = 1/(1+np.exp(-np.vstack(all_logits)))
    Y = np.vstack(all_targets)
    aucs = []
    for j in range(Y.shape[1]):
        y_true = Y[:, j]
        if y_true.sum() == 0 or (1 - y_true).sum() == 0:
            continue
        aucs.append(roc_auc_score(y_true, P[:, j]))
    return float(np.mean(aucs)) if aucs else 0.0

print("✅ Eval function defined")

✅ Eval function defined


In [12]:
# PREPARE TRAINING DATAFRAME & VARIABLES
import torch
import numpy as np
from pathlib import Path

device = torch.device(CFG["device"])

# Count mel files available
mel_root = Path("/kaggle/working/mels")
available_mels = set(f.stem.replace('.npy', '') for f in mel_root.glob("*.npy"))

print(f"Available mel files: {len(available_mels)}")

# Build training dataframe from train_audio
train_df = df.copy()
train_df["filename"] = train_df["filename"].apply(lambda x: x.replace("/", "_"))

# Filter to only rows with available mels
train_df = train_df[train_df["filename"].isin(available_mels)]

print(f"Training samples with mels: {len(train_df)}")
print(f"Sample columns: {train_df.columns.tolist()}")

# Setup species variables
trained_species = species
n_classes = len(species)

# Count occurrences of each species
species_counts = {sp: 0 for sp in species}
for idx_, row in train_df.iterrows():
    primary = str(row["primary_label"])
    if primary in species_counts:
        species_counts[primary] += 1
    
    # Count secondary labels if present
    secondary = row.get("secondary_labels", [])
    if pd.notna(secondary):
        secondary_list = parse_secondary(secondary)
        for sp in secondary_list:
            if sp in species_counts:
                species_counts[sp] += 1

print(f"\n✅ Training setup complete:")
print(f"  - Device: {device}")
print(f"  - Classes: {n_classes}")
print(f"  - Train samples: {len(train_df)}")
print(f"  - Species with data: {sum(1 for c in species_counts.values() if c > 0)}")


Available mel files: 36068
Training samples with mels: 36587
Sample columns: ['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author', 'license', 'rating', 'url', 'filename', 'collection']

✅ Training setup complete:
  - Device: cuda
  - Classes: 206
  - Train samples: 36587
  - Species with data: 206


In [13]:
# 5-FOLD CROSS-VALIDATION TRAINING
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
import json
from datetime import datetime

CFG = {
    "seconds": 10,
    "sr": 16000,
    "n_mels": 64,
    "n_fft": 1024,
    "hop": 320,
    "fmin": 60,
}

TRAINING_CFG = {
    "epochs": 15,
    "batch_size": 32,
    "lr": 1e-3,
    "early_stopping_patience": 5,
    "class_weight_factor": 3.0,  # Increased from 2.0 for stronger class balancing
}

all_scores = []
fold_scores = {}
ALL_VAL_PREDS = []  # Collect validation predictions for threshold analysis
ALL_VAL_TARGETS = []  # Collect validation targets for threshold analysis

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    print(f"\n🔄 Fold {fold_idx + 1}/5")
    
    train_fold = train_df.iloc[train_idx]
    val_fold = train_df.iloc[val_idx]
    
    # Datasets
    train_ds = ClipDataset(train_fold, MEL_ROOT, CFG, train=True)
    val_ds = ClipDataset(val_fold, MEL_ROOT, CFG, train=False)
    
    # DataLoaders
    train_loader = DataLoader(
        train_ds, batch_size=TRAINING_CFG["batch_size"], shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=TRAINING_CFG["batch_size"], shuffle=False, num_workers=0
    )
    
    # Model
    model = ResNet50Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    
    # Class weighting (increased factor for stronger class balancing)
    pos_weight = torch.ones(n_classes).to(device)
    for i, sp in enumerate(trained_species):
        factor = TRAINING_CFG["class_weight_factor"]
        pos_weight[i] = len(train_df) / (factor * species_counts[sp])
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = AdamW(model.parameters(), lr=TRAINING_CFG["lr"])
    
    # Training loop with early stopping
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(TRAINING_CFG["epochs"]):
        # Train
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        print(f"  Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}", end="")
        
        # Early stopping
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f"model_fold_{fold_idx}.pt")
            print(" ✅")
        else:
            patience_counter += 1
            print()
            if patience_counter >= TRAINING_CFG["early_stopping_patience"]:
                print(f"  ⛔ Early stopping at epoch {epoch+1}")
                break
    
    # Collect validation predictions from best model for threshold analysis
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            probs = torch.sigmoid(logits).cpu().numpy()
            targets = y.cpu().numpy()
            ALL_VAL_PREDS.append(probs)
            ALL_VAL_TARGETS.append(targets)
    
    fold_scores[f"fold_{fold_idx}"] = best_loss
    all_scores.append(best_loss)
    print(f"  Best Val Loss: {best_loss:.4f}")

print("\n📊 Cross-Validation Results:")
print(f"  Mean Val Loss: {np.mean(all_scores):.4f} ± {np.std(all_scores):.4f}")
print(f"  Fold Scores: {fold_scores}")
print(f"\n✅ Collected {len(ALL_VAL_PREDS)} batches of validation predictions")


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



🔄 Fold 1/5
  Epoch  1 | Train Loss: 0.8411 | Val Loss: 0.8461 ✅
  Epoch  2 | Train Loss: 0.8105 | Val Loss: 0.8027 ✅
  Epoch  3 | Train Loss: 0.7795 | Val Loss: 0.7873 ✅
  Epoch  4 | Train Loss: 0.7659 | Val Loss: 0.7818 ✅
  Epoch  5 | Train Loss: 0.7591 | Val Loss: 0.7803 ✅
  Epoch  6 | Train Loss: 0.7552 | Val Loss: 0.7806
  Epoch  7 | Train Loss: 0.7527 | Val Loss: 0.7819
  Epoch  8 | Train Loss: 0.7510 | Val Loss: 0.7837
  Epoch  9 | Train Loss: 0.7499 | Val Loss: 0.7858
  Epoch 10 | Train Loss: 0.7490 | Val Loss: 0.7881
  ⛔ Early stopping at epoch 10
  Best Val Loss: 0.7803

🔄 Fold 2/5
  Epoch  1 | Train Loss: 0.8469 | Val Loss: 0.8367 ✅
  Epoch  2 | Train Loss: 0.8128 | Val Loss: 0.7897 ✅
  Epoch  3 | Train Loss: 0.7819 | Val Loss: 0.7713 ✅
  Epoch  4 | Train Loss: 0.7687 | Val Loss: 0.7627 ✅
  Epoch  5 | Train Loss: 0.7619 | Val Loss: 0.7584 ✅
  Epoch  6 | Train Loss: 0.7584 | Val Loss: 0.7556 ✅
  Epoch  7 | Train Loss: 0.7559 | Val Loss: 0.7540 ✅
  Epoch  8 | Train Loss: 0.754

In [14]:
# COMPUTE PER-SPECIES OPTIMAL THRESHOLDS (OPTIONAL)
print("\n" + "="*60)
print("THRESHOLD ANALYSIS")
print("="*60)

combined_preds = np.vstack(ALL_VAL_PREDS)
combined_targets = np.vstack(ALL_VAL_TARGETS)

# NOTE: Per-species thresholds hurt performance in Phase 1 (0.648 → 0.559)
# Using simple approach: threshold=0.5 for all species
# This matches the original winning 0.648 configuration

optimal_thresholds = {sp: 0.5 for sp in species}

# Save thresholds (uniform 0.5)
with open("/kaggle/working/optimal_thresholds.json", "w") as f:
    json.dump(optimal_thresholds, f)

print(f"\n✅ Using uniform thresholds: 0.5 for all {len(optimal_thresholds)} species")
print(f"Reasoning: Per-species F1-optimization hurt validation performance")
print(f"Strategy: Return to winning 0.648 configuration (uniform thresholds)")
print(f"\n💾 Saved to: /kaggle/working/optimal_thresholds.json")


THRESHOLD ANALYSIS

✅ Using uniform thresholds: 0.5 for all 206 species
Reasoning: Per-species F1-optimization hurt validation performance
Strategy: Return to winning 0.648 configuration (uniform thresholds)

💾 Saved to: /kaggle/working/optimal_thresholds.json
